# Baseline Models for Rent Price Prediction

This notebook demonstrates the creation of baseline models for rent price prediction.

**Key steps:**
- Data loading and feature selection
- Log-transforming the target variable (`price`) for better model performance
- Using cross-validation to evaluate models
- Comparing baseline models using RMSLE metric (on original price scale), because RMSLE penalized the underestimate more than overestimate

## Imports

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

import pandas as pd
import lightgbm as lgb
import xgboost as xgb

from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

from src.models.model_evaluation import run_cv

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)

## Load Data and Select Features

In [2]:
data = pd.read_csv('../data/processed/train_df.csv')

In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9140 entries, 0 to 9139
Data columns (total 29 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   9140 non-null   int64  
 1   price                9140 non-null   int64  
 2   address              9140 non-null   object 
 3   coordinates          9032 non-null   object 
 4   region               9140 non-null   object 
 5   subway               7375 non-null   object 
 6   rooms                9140 non-null   int64  
 7   footage              9140 non-null   object 
 8   floor                9140 non-null   int64  
 9   features             9140 non-null   object 
 10  residential          4367 non-null   object 
 11  neighborhood         9140 non-null   object 
 12  description          9140 non-null   object 
 13  detail               9140 non-null   object 
 14  attributes           5501 non-null   object 
 15  full_area            9140 non-null   f

In [4]:
black_list = ['id', 'price', 'price_bin']

numerical_feats = [col for col in data.select_dtypes(exclude='object').columns
                   if col not in black_list]

In [5]:
data[numerical_feats].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9140 entries, 0 to 9139
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   rooms                9140 non-null   int64  
 1   floor                9140 non-null   int64  
 2   full_area            9140 non-null   float64
 3   living_area          9137 non-null   float64
 4   kitchen_area         9134 non-null   float64
 5   num_storeys          9140 non-null   int64  
 6   lon                  9032 non-null   float64
 7   lat                  9032 non-null   float64
 8   unknown_living_area  9140 non-null   int64  
dtypes: float64(5), int64(4)
memory usage: 642.8 KB


In [6]:
X = data[numerical_feats]
y = data['price']

## Baseline Models

In [7]:
results = {}

### DummyRegressor (mean)

In [8]:
model = DummyRegressor(strategy='mean')
mean_score = run_cv(model, X, y)
results['DummyRegressor-mean'] = mean_score

[Fold 0] train_rmsle: 0.7376, val_rmsle: 0.7320
[Fold 1] train_rmsle: 0.7351, val_rmsle: 0.7373
[Fold 2] train_rmsle: 0.7344, val_rmsle: 0.7385
RMSLE: 0.7359 ± 0.0028


### DummyRegressor (median)

In [9]:
model = DummyRegressor(strategy='median')
median_score = run_cv(model, X, y)
results['DummyRegressor-median'] = median_score

[Fold 0] train_rmsle: 0.7414, val_rmsle: 0.7368
[Fold 1] train_rmsle: 0.7404, val_rmsle: 0.7387
[Fold 2] train_rmsle: 0.7378, val_rmsle: 0.7440
RMSLE: 0.7398 ± 0.0030


### Linear Regression

In [10]:
pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

lr_score = run_cv(pipeline, X, y)
results['LinearRegression'] = lr_score

[Fold 0] train_rmsle: 0.4860, val_rmsle: 0.4774
[Fold 1] train_rmsle: 0.4776, val_rmsle: 0.4944
[Fold 2] train_rmsle: 0.4845, val_rmsle: 0.4804
RMSLE: 0.4841 ± 0.0074


### ExtraTreesRegressor

In [11]:
pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('model', ExtraTreesRegressor(n_jobs=-1, random_state=7))
])

et_score = run_cv(pipeline, X, y)
results['ExtraTreesRegressor'] = et_score

[Fold 0] train_rmsle: 0.0060, val_rmsle: 0.2927
[Fold 1] train_rmsle: 0.0062, val_rmsle: 0.3039
[Fold 2] train_rmsle: 0.0071, val_rmsle: 0.3051
RMSLE: 0.3006 ± 0.0056


### RandomForestRegressor

In [12]:
pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('model', RandomForestRegressor(n_jobs=-1, random_state=7))
])

rf_score = run_cv(pipeline, X, y)
results['RandomForestRegressor'] = rf_score

[Fold 0] train_rmsle: 0.1109, val_rmsle: 0.2841
[Fold 1] train_rmsle: 0.1086, val_rmsle: 0.2970
[Fold 2] train_rmsle: 0.1076, val_rmsle: 0.2967
RMSLE: 0.2926 ± 0.0060


### XGBoost

In [13]:
parameters_xgb = {
    'verbosity': 1,
    'seed': 7,
}
model = xgb.XGBRegressor(**parameters_xgb)

xgb_score = run_cv(model, X, y)
results['XGBRegressor'] = xgb_score

[Fold 0] train_rmsle: 0.1435, val_rmsle: 0.2761
[Fold 1] train_rmsle: 0.1408, val_rmsle: 0.2890
[Fold 2] train_rmsle: 0.1357, val_rmsle: 0.2933
RMSLE: 0.2861 ± 0.0073


### LightGBM

In [14]:
parameters_lgb = {
    'random_state': 7,
    'verbose': 0,
}

model = lgb.LGBMRegressor(**parameters_lgb)

lgb_score = run_cv(model, X, y)
results['LGBMRegressor'] = lgb_score

[Fold 0] train_rmsle: 0.2309, val_rmsle: 0.2873
[Fold 1] train_rmsle: 0.2287, val_rmsle: 0.2910
[Fold 2] train_rmsle: 0.2256, val_rmsle: 0.2933
RMSLE: 0.2905 ± 0.0025


## Summary Table

In [15]:
df_results = pd.DataFrame(results.items(), columns=['Model', 'RMSLE'])
df_results = df_results.sort_values('RMSLE')
df_results.style.highlight_min(subset=['RMSLE'], color='tan')

,Model,RMSLE
5,XGBRegressor,0.286146
6,LGBMRegressor,0.290497
4,RandomForestRegressor,0.292598
3,ExtraTreesRegressor,0.300555
2,LinearRegression,0.484070
0,DummyRegressor-mean,0.735939
1,DummyRegressor-median,0.739846


- We established several baseline models for rent price prediction using only numerical features and log-transformed target values.
- Tree-based models (RandomForest, ExtraTrees, XGBoost, LightGBM) significantly outperform simple baselines (Dummy, Linear Regression) in terms of RMSLE.
- The best baseline RMSLE is achieved by XGBoost (0.286), followed closely by LightGBM (0.290) and RandomForest (0.293).

**Conclusions:**
- Keep DummyRegressors as baseline references for future experiments.
- Perform feature engineering, incorporate categorical, text, and geospatial features.
- Proceed with hyperparameter tuning for XGB, LGBM and Random Forest as the leading baselines to further improve performance.